# 01 – Data Generation

This notebook shows how to generate a small synthetic LPBF dataset that can be used to train the Physics-Informed Neural Network (PINN).

The repository ships with `src/generate_synthetic_data.py`, which contains `SyntheticDataGenerator`. This class produces physics-inspired empirical outputs for three quality metrics:

1. **Residual stress** (MPa)
2. **Porosity** (volume fraction)
3. **Geometric accuracy** (ratio)

The generator also supports `LPBFDataPreprocessor` for converting real FEA outputs (`.h5`, `.odb`, `.vtk`) into the same HDF5 training format. Because real FEA data requires external licenses, this walkthrough uses only synthetic data.

In [ ]:
import os
import sys

# Make the repository's src/ package importable from this notebook
# In a notebook __file__ is not defined, so we assume the notebook is run from notebooks/
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.insert(0, project_root)

# Scripts in src/ expect to be run from the repository root, so switch CWD
os.chdir(project_root)
print(f"Working directory: {os.getcwd()}")
print(f"Project root: {project_root}")

## 1.1 Generate a small synthetic dataset

We use a tiny configuration so the notebook runs quickly. The default CLI uses 50 scan vectors with 64 points each; here we use 20 scan vectors with 27 points each (a `3×3×3` spatial grid).

In [ ]:
from src.generate_synthetic_data import SyntheticDataGenerator

config_path = os.path.join('data', 'params.yaml')
generator = SyntheticDataGenerator(config_path)

dataset_path = generator.generate(
    n_scan_vectors=20,
    n_points_per_vector=27,  # must be a perfect cube because of the internal grid
)
print(f"\nDataset written to: {dataset_path}")

## 1.2 Inspect the generated HDF5 file

The processed dataset contains train/val/test splits, leakage-free scan-vector IDs, and standard-scaler parameters.

In [ ]:
import h5py

with h5py.File(dataset_path, 'r') as f:
    print('Top-level groups:', list(f.keys()))
    print('\nMetadata:')
    for key, value in f['metadata'].attrs.items():
        print(f"  {key}: {value}")
    print('\nDatasets per split:')
    for split in ['train', 'val', 'test']:
        print(f"  {split}: inputs {f[split]['inputs'].shape}, outputs {f[split]['outputs'].shape}")
    print('\nParameter names:', f['metadata']['parameter_names'][:])

## 1.3 Look at one scan vector and its output distribution

Each input row concatenates `[scan_vector, x, y, z, t]`. The outputs are `[residual_stress, porosity, geometric_accuracy]`.

In [ ]:
import pandas as pd

with h5py.File(dataset_path, 'r') as f:
    inputs = f['train/inputs'][:]
    outputs = f['train/outputs'][:]

print('First training input (parameters + coordinates + time):', inputs[0])
print('First training output (stress, porosity, accuracy):', outputs[0])

df = pd.DataFrame(outputs, columns=['residual_stress_MPa', 'porosity', 'geometric_accuracy'])
df.describe()

## 1.4 Optional: preprocessing real FEA data

`LPBFDataPreprocessor` converts raw FEA simulation files into the same HDF5 layout. This step is shown here only for completeness; it requires real FEA `.h5` files under `data/raw/` and will report that no files are found if run in synthetic-only mode.

In [ ]:
from src.preprocessing import LPBFDataPreprocessor

preprocessor = LPBFDataPreprocessor(config_path)
raw_files = preprocessor.find_raw_files(pattern='*.h5')
print(f"Found {len(raw_files)} raw FEA files to preprocess.")

if raw_files:
    h5_path = preprocessor.process_all_files()
    print(f"Preprocessed dataset saved to: {h5_path}")
else:
    print('Skipping preprocessing because no raw FEA files are present.')

## Next step

Proceed to `02_training.ipynb` to train a PINN surrogate on this dataset.